In [1]:
!pip install deep-sort-realtime

Defaulting to user installation because normal site-packages is not writeable


In [15]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

In [18]:
# Load YOLO model
model = YOLO(r"..\fine_tunning_3\runs\detect\train\weights\best.pt")

In [19]:
# Initialize DeepSORT
tracker = DeepSort(
    max_age=70,
    n_init=5,
    max_iou_distance=0.7,
    max_cosine_distance=0.3,
    bgr=True
)

In [20]:
# Open video
cap = cv2.VideoCapture(r"..\video1.mp4")

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    r".\fine_tune_3\tracking_result1.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)

In [23]:
def get_color_histogram(crop, bins=(8, 8, 8)):
    h, w = crop.shape[:2]
    upper_body = crop[: int(h * 0.5), :]
    hsv = cv2.cvtColor(upper_body, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist(
        [hsv],
        [0, 1, 2],
        None,
        bins,
        [0, 180, 0, 256, 0, 256]
    )
    hist = cv2.normalize(hist, hist).flatten()

    return hist

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

In [21]:
player_features = {}      # track_id -> histogram
team_memory = {}          # track_id -> Team 1 / Team 2
clustering_done = False

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # ---------------- YOLO ----------------
    results = model(frame, conf=0.3, iou=0.5)

    detections = []

    for result in results:

        for box in result.boxes:

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            conf = float(box.conf[0])
            cls = int(box.cls[0])

            w = x2 - x1
            h = y2 - y1

            detections.append(
                (
                    [x1, y1, w, h],
                    conf,
                    cls
                )
            )

    # ---------------- DeepSORT ----------------
    tracks = tracker.update_tracks(
        detections,
        frame=frame
    )

    # ---------------- Tracking ----------------
    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        class_id = track.get_det_class()
        class_name = model.names[class_id]

        l, t, r, b = map(int, track.to_ltrb())

        # Empêcher les coordonnées négatives
        l = max(0, l)
        t = max(0, t)
        r = min(frame.shape[1], r)
        b = min(frame.shape[0], b)

        # ---------------- Player Feature Extraction ----------------
        if class_name == "Player":

            if track_id not in player_features:

                player_crop = frame[t:b, l:r]

                if player_crop.size != 0:

                    player_hist = get_color_histogram(player_crop)

                    player_features[track_id] = player_hist

        # ---------------- Draw ----------------

        if track_id in team_memory:
            label = f"{team_memory[track_id]} | ID:{track_id}"
        else:
            label = f"{class_name} | ID:{track_id}"

        cv2.rectangle(
            frame,
            (l, t),
            (r, b),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            label,
            (l, t - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    # ---------------- KMeans ----------------

    if not clustering_done and len(player_features) >= 18:

        ids = list(player_features.keys())

        features = np.array(list(player_features.values()))

        kmeans = KMeans(
            n_clusters=2,
            random_state=42,
            n_init="auto"
        )

        labels = kmeans.fit_predict(features)

        for track_id, cluster in zip(ids, labels):

            if cluster == 0:
                team_memory[track_id] = "Team A"
            else:
                team_memory[track_id] = "Team B"

        clustering_done = True

        print("Players clustered successfully!")

    out.write(frame)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 18 Players, 158.6ms
Speed: 9.0ms preprocess, 158.6ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 93.1ms
Speed: 7.2ms preprocess, 93.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 17 Players, 67.7ms
Speed: 2.6ms preprocess, 67.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 17 Players, 78.3ms
Speed: 2.8ms preprocess, 78.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 17 Players, 80.6ms
Speed: 2.7ms preprocess, 80.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 79.0ms
Speed: 3.3ms preprocess, 79.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players, 79.9ms
Speed: 3.3ms preprocess, 79.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 18 Players